# Fase 1 — Extracción y Limpieza Estructural
### Versión Google Colab (`.ipynb` ejecutable de principio a fin)

Esta es la versión para Colab del script `fase1_limpieza_estructural.py`: **misma
clase `LimpiadorNoticias`, mismas reglas de limpieza** (mojibake en dos capas,
patrones de navegación/suscripción/boilerplate por portal, protección estricta de los
`\n\n` de párrafo), pero sustituyendo la interfaz de línea de comandos (`argparse`)
por celdas pensadas para ejecutarse en Colab: montaje de Google Drive (o
`files.upload()` si prefieres subir el archivo directamente) y variables simples de
entrada/salida en vez de argumentos de terminal.

**Cómo usarlo:** Entorno de ejecución → Ejecutar todas. Si no tienes el archivo en tu
Drive, la celda 2 te lo pedirá con un selector de subida.


## 1. Carga del archivo JSON

Sube directamente tu archivo JSON de noticias scrapeadas (sin necesidad de montar
Google Drive). Al ejecutar la celda se abrirá el selector de subida de Colab.


In [ ]:
# --- Detección de entorno (Colab vs. local) ----------------------------------
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("No estamos en Google Colab: se omite el selector de subida.")

import os

INPUT_JSON = None

if IN_COLAB:
    print("Sube tu archivo JSON de noticias scrapeadas:")
    subido = files.upload()
    if subido:
        INPUT_JSON = list(subido.keys())[0]
        print(f"Archivo recibido: {INPUT_JSON}")
    else:
        raise RuntimeError("No se subió ningún archivo. Vuelve a ejecutar la celda para intentarlo de nuevo.")
else:
    # Fuera de Colab: usa un archivo local ya presente en el directorio actual.
    INPUT_JSON = "noticias_scrapeadas.json"
    if not os.path.exists(INPUT_JSON):
        raise FileNotFoundError(
            f"No se encontró '{INPUT_JSON}' localmente. Coloca tu archivo de entrada "
            f"en el directorio actual o ajusta la variable INPUT_JSON manualmente."
        )
    print(f"Usando archivo local: {INPUT_JSON}")

# Nombre del archivo de salida: se deriva del de entrada añadiendo un sufijo,
# para no sobrescribir el original.
_nombre_base, _ext = os.path.splitext(INPUT_JSON)
if not _ext:
    _ext = ".json"
OUTPUT_JSON = f"{_nombre_base}_fase1{_ext}"
print(f"Salida: {OUTPUT_JSON}")


## 2. Función core de limpieza (CRÍTICA)

### 2.1 Mojibake — dos capas complementarias

- **Capa "rápida"** (`reparar_mojibake_global`): recodifica el texto completo de
  `latin1` a `utf-8` dentro de un `try/except`. Solo corrige algo cuando **todo** el
  documento está mal codificado; en un artículo real (mojibake solo en fragmentos
  sueltos) casi nunca tiene efecto por sí sola.
- **Capa robusta** (`reparar_mojibake_fragmentos`): repara solo los fragmentos con
  pinta de mojibake (`Ã©`, `â€™`...), dejando intacto el resto del texto. Es la que
  hace el trabajo real en la práctica.

Se aplican siempre las dos, en ese orden.


In [ ]:
import re
import html

# --- 2.1a Reparación "rápida" de mojibake: recodificación completa -----------
def reparar_mojibake_global(text: str) -> str:
    """Intento rápido de reparación de mojibake sobre el texto completo.
    Solo tiene efecto cuando TODO el documento está mal codificado; en
    documentos reales (mojibake solo en fragmentos) no hace nada, y por eso
    se complementa siempre con reparar_mojibake_fragmentos()."""
    if not text:
        return text
    try:
        return text.encode("latin1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text


# --- 2.1b Reparación robusta de mojibake: fragmento a fragmento --------------
# Patrón de un fragmento mojibake: empieza por Ã/Â/â y va seguido de uno o más
# caracteres del rango típico que resulta de decodificar bytes UTF-8 (0x80-0xFF)
# como cp1252 (comillas tipográficas, guiones largos, €, ™, vocales acentuadas...).
_MOJIBAKE_RUN = re.compile(
    "[ÃÂâ][€‚ƒ„…†‡ˆ‰Š‹ŒŽ‘’“”•–—˜™š›œžŸ\u00A0-\u00FF]+"
)

def _reparar_fragmento(match: "re.Match") -> str:
    frag = match.group(0)
    try:
        return frag.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return frag  # no era mojibake real, se deja tal cual

def reparar_mojibake_fragmentos(text: str) -> str:
    """Repara texto UTF-8 que fue re-codificado incorrectamente como
    cp1252/latin-1 en algún punto del pipeline de scraping (p. ej.
    "Japanâ€™s" -> "Japan\'s", "ComunicaciÃ³n" -> "Comunicación")."""
    if not text or not _MOJIBAKE_RUN.search(text):
        return text
    return _MOJIBAKE_RUN.sub(_reparar_fragmento, text)


def reparar_mojibake(text: str) -> str:
    """Aplica las dos capas de reparación de mojibake, en orden."""
    text = reparar_mojibake_global(text)
    text = reparar_mojibake_fragmentos(text)
    return text


### 2.2 Limpieza del título

In [ ]:
def clean_titulo(titulo: str, contenido: str = "") -> str:
    """Limpia el campo titulo de artefactos del scraper."""
    if not titulo:
        return titulo

    titulo = reparar_mojibake(titulo)
    titulo = html.unescape(titulo)
    titulo = re.sub(r"<!--.*?-->", "", titulo)
    titulo = re.sub(
        r"\s*-\s*(?:Barron[\u2019\u2018\'\']?s|Reuters|Bloomberg|CNBC|"
        r"MarketWatch|El\s+Pa[íi]s|El\s+Mundo|Expans[ií]on|Cinco\s+D[ií]as)\s*$",
        "", titulo, flags=re.I,
    )
    titulo = re.sub(r"\s*\|.*$", "", titulo)

    if re.match(r"^\.[A-Za-z_]", titulo) and "{" in titulo:
        m = re.match(r"^([^\n\\]+?)(?:\s*\\+\s*-?\s*MarketWatch)?\s*(?:\n|\\)", contenido or "")
        titulo = re.sub(r"\s*[-\\]+\s*MarketWatch\s*$", "", m.group(1), flags=re.I).strip() if m else ""

    if titulo.strip() == "Expansión - Edicion Impresa":
        m = re.search(r"\n# ([^\n]+)\n", contenido or "")
        titulo = m.group(1).strip() if m else titulo

    return titulo.strip()


### 2.3 Ruido genérico común a todas las fuentes

In [ ]:
def clean_content(text: str, titulo: str = "") -> str:
    """Pipeline base: elimina ruido genérico común a todas las fuentes."""
    text = reparar_mojibake(text)
    text = html.unescape(text)
    text = re.sub(r"\ufffd+", "", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    text = re.sub(r"\[([^\]]{1,120})\]\(https?://[^)]+\)", r"\1", text)
    text = re.sub(r"\[\]\(https?://[^)]+\)", "", text)
    text = re.sub(r"^\s*https?://\S+\s*$", "", text, flags=re.M)
    text = re.sub(r"https?://\S+", "", text)

    _NAV_PATTERNS = [
        r"(?:Accessibility help\s*)?Skip to (?:Main Content|navigation|main content|content|Search|footer)",
        r"Add to myFT\b[^\n]*",
        r"Remove from myFT\b[^\n]*",
        r"Get instant alerts for this topic[^\n]*",
        r"Manage your delivery channels here[^\n]*",
        r"Show articles\s*",
        r"Stay informed with free updates[^\n]*",
        r"Simply sign up to the[^\n]*Digest[^\n]*",
        r"Log in\s*\nSearch\s*\nSearch\s*",
        r"Try our new[^\n]*AI-powered search[^\n]*beta",
    ]
    for pat in _NAV_PATTERNS:
        text = re.sub(pat, "", text, flags=re.I)

    _SOCIAL = re.compile(
        r"^\s*(?:"
        r"(?:Share|Compartir)(?: en| on| via)?(?:\s+(?:Twitter|Facebook|LinkedIn|"
        r"Whatsapp|X|Instagram|Email|email))?"
        r"|Enviar por email"
        r"|Desplegar Redes Sociales"
        r"|Añadir \w+ en Google"
        r"|on (?:x|facebook|linkedin|whatsapp|twitter)\s*\(opens in a new window\)"
        r")\s*$",
        re.M | re.I,
    )
    text = _SOCIAL.sub("", text)

    text = re.sub(
        r"^[^\n]{0,30}(?:Subscribe (?:Now|for full access|to \w+(?:\s+\w+)?)|\"SUBSCRIBE NOW|Suscr[íi]bete(?:\s+ahora)?|SUSCR[ÍI]BETE)\s*$",
        "", text, flags=re.M | re.I,
    )
    text = re.sub(r"Continue reading this article with a[^\n]+\n(?:[^\n]*SUBSCRIBE[^\n]*)?", "", text, flags=re.I)
    text = re.sub(r"Subscribe to \w+(?:\s+\w+)? (?:PRO )?for[^\n]+(?:program|access|content|benefits)[^\n]*\n?", "", text, flags=re.I)

    text = re.sub(r"^[^\n]{0,30}(?:Cookie Notice|Política de (?:cookies|privacidad)|Privacy Policy|Uso de cookies)[^\n]*$", "", text, flags=re.M | re.I)

    def _is_nav_bullet(line: str) -> bool:
        s = line.strip()
        for pfx in ("- ", "* ", "• "):
            if s.startswith(pfx):
                inner = s[len(pfx):].strip()
                return len(inner) <= 100 and not re.search(r"[.!?]\s+\w", inner)
        return False

    lines = text.split("\n")
    out_lines, i = [], 0
    while i < len(lines):
        j = i
        while j < len(lines) and _is_nav_bullet(lines[j]):
            j += 1
        if j - i >= 4:
            i = j
        else:
            out_lines.append(lines[i])
            i += 1
    text = "\n".join(out_lines)

    _UI_HEADINGS = re.compile(
        r"^#{1,3}\s*(?:Topics|Memberships|Tools|Customer Service|Network|"
        r"Newsstand|Latest|More|Newsletter|Follow us|Connect|Sections)\s*$",
        re.M | re.I,
    )
    text = _UI_HEADINGS.sub("", text)

    _UI_LABELS = re.compile(
        r"^\s*(?:Save|Resize|Reprints?|Follow|Listen|Search|"
        r"Guardar|Imprimir|Comentar|PREMIUM|"
        r"By\s*$|Por\s*$|Autor[:\s]*$|Tags?[:\s]*$|\(\d+\s*min\))\s*$",
        re.M,
    )
    text = _UI_LABELS.sub("", text)

    if titulo:
        title_esc = re.escape(titulo[:70].strip())
        text = re.sub(r"^\s*" + title_esc + r"[^\n]*\n", "", text, count=1)

    text = re.sub(r"[ \t]+$", "", text, flags=re.M)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


### 2.4 Artefactos específicos por portal

In [ ]:
def clean_content_extra(text: str, titulo: str = "") -> str:
    """Segunda fase de limpieza: artefactos específicos por portal."""
    text = re.sub(r"^Oops,\s*something went wrong\s*\n+", "", text, flags=re.I)
    if titulo:
        esc = re.escape(titulo[:80].strip())
        text = re.sub(r"^#+\s*" + esc + r"[^\n]*\n+", "", text, count=1, flags=re.I)

    if text.startswith("_\n\n_"):
        meses = r"(?:ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)"
        m = re.search(rf"\d{{1,2}} {meses} \d{{4}} - \d{{2}}:\d{{2}}(?:CET|CEST)\s*\n+", text[:700], re.I)
        if m:
            text = text[m.end():]
        else:
            text = re.sub(r"^(?:_[ \t]*\n\n)+", "", text)
            text = re.sub(r"^(?:Crónica de la Bolsa|Mercados [Ff]inancieros|Economía|Empresas|Opinión|Bolsa)\s*\n\n?", "", text)

    if "Ir a Expansion.com" in text:
        m = re.search(r"\n(# [^#\n][^\n]+)\n", text)
        if m:
            text = text[m.start() + 1:]

    if "**Site SearchClear" in text:
        m = re.search(r"\n(## [^\n]+)\n", text)
        if m:
            text = text[m.start() + 1:]

    if "Líder mundial en español" in text:
        m = re.search(r"\n(## [^\n]+)\n", text)
        if m:
            text = text[m.start() + 1:]

    text = re.sub(r"^\[Skip Navigation\]\(#\w+\)\s*\n", "", text, flags=re.M)
    text = re.sub(r"^BREAKING\s*\n", "", text, flags=re.M)
    text = re.sub(r"^!\[[^\]]*\]\([^\)]+\)\s*\n?", "", text, flags=re.M)
    text = re.sub(r"^The coverage on this live blog has ended[^\n]*\n+", "", text, flags=re.M | re.I)
    text = re.sub(r"^This article was originally published on[^\n]+\n", "", text, flags=re.M | re.I)
    text = re.sub(r"^This is a paid press release\.[^\n]*\n+", "", text, flags=re.M | re.I)
    text = re.sub(r"^Ir a los comentarios\s*$", "", text, flags=re.M | re.I)

    text = re.sub(r"[ \t]+$", "", text, flags=re.M)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


### 2.5 Boilerplate de cabecera/pie por portal

In [ ]:
def clean_content_footer(text: str) -> str:
    """Tercera fase: elimina boilerplate de cabecera/pie específico de portales
    (El País, CNBC, FT, Barron\'s/Dow Jones, MarketWatch/Fusion Media, etc.).

    Estrategia:
      - Patrones que casi siempre aparecen como PIE del artículo se usan para
        truncar el texto en ese punto, pero solo si el match cae en el último
        tramo del documento (para no comernos contenido real si el patrón
        aparece antes por coincidencia).
      - Patrones que aparecen como incisos sueltos a mitad de artículo se
        eliminan como línea/frase puntual, sin truncar nada.
    """
    if not text:
        return text

    text = re.sub(
        r"\s*CNBC (?:PRO|Pro) subscribers can read more here\.",
        "", text,
    )
    text = re.sub(
        r"Roula Khalaf, Editor of the FT, selects her favourite stories in this weekly newsletter\.?\s*",
        "", text,
    )
    text = re.sub(
        r"^\s*Añadir [\wÁÉÍÓÚáéíóúñÑ]+(?:\s[\wÁÉÍÓÚáéíóúñÑ]+)? en Google\s*$",
        "", text, flags=re.M,
    )
    text = re.sub(
        r"#{0,4}\s*Noticias [Rr]elacionadas\s*\n+(?:-[^\n]+\n*)+",
        "\n\n", text, flags=re.I,
    )

    if "Subscriber Agreement and by copyright law" in text:
        m = re.search(r"\n(# [^#\n][^\n]+)\n", text)
        if m:
            text = text[m.start() + 1:]

    _FOOTER_MARKERS = [
        r"##\s*Archivado En",
        r"Choose CNBC as your preferred source on Google[^\n]*",
        r"Copyright\s*©[^\n]*",
        r"©\s*\d{4}[-–]\d{4}\s*-\s*Fusion Media Limited[^\n]*",
        r"Related Articles\s*\n",
        r"Noticias Relacionadas\s*\n",
        r"Sigue toda la informaci[oó]n de[^\n]* en Facebook",
        r"Suscr[ií]bete en El Pa[ií]s para participar",
        r"Rellena tu nombre y apellido",
        r"### Destacados\s*\n\s*### Servicios",
    ]
    _FOOTER_MARKERS_ESPECIFICOS = [
        (r"####\s*Partner Center", 0.4),
        (r"Back To Top", 0.4),
    ]

    cut_at = None
    for pat in _FOOTER_MARKERS:
        m = re.search(pat, text, flags=re.I)
        if m and len(text) > 0 and (m.start() / len(text)) >= 0.6:
            if cut_at is None or m.start() < cut_at:
                cut_at = m.start()
    for pat, umbral in _FOOTER_MARKERS_ESPECIFICOS:
        m = re.search(pat, text, flags=re.I)
        if m and len(text) > 0 and (m.start() / len(text)) >= umbral:
            if cut_at is None or m.start() < cut_at:
                cut_at = m.start()
    if cut_at is not None:
        text = text[:cut_at]

    text = re.sub(r"[ \t]+$", "", text, flags=re.M)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


### 2.6 Espacios horizontales y "regla de oro" de los saltos de línea

In [ ]:
def normalizar_espacios_horizontales(text: str) -> str:
    """Colapsa espacios y tabulaciones múltiples a uno solo, SIN tocar los
    saltos de línea (la regex [ \t]+ nunca hace match con \'\\n\')."""
    if not text:
        return text
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"[ \t]*\n[ \t]*", "\n", text)
    return text


def normalizar_saltos_de_linea(text: str) -> str:
    """REGLA DE ORO: nunca deja más de un salto de línea vacío seguido.
    3 o más \'\\n\' consecutivos se normalizan siempre a exactamente \'\\n\\n\',
    protegiendo la estructura de párrafos para el SentenceSplitter."""
    if not text:
        return text
    return re.sub(r"\n{3,}", "\n\n", text)


### 2.7 Clase principal: `LimpiadorNoticias`

In [ ]:
class LimpiadorNoticias:
    """Encapsula el pipeline completo de la Fase 1 (Extracción y Limpieza
    Estructural)."""

    def limpiar_articulo(self, titulo: str, contenido: str):
        """Limpia titulo + contenido de un artículo.

        Orden de aplicación (importa):
          1. Limpieza del título (necesaria porque clean_content usa el
             título limpio para eliminar la línea que lo repite al inicio
             del cuerpo).
          2. Ruido genérico común a todas las fuentes.
          3. Artefactos específicos de portal (segunda pasada).
          4. Boilerplate de cabecera/pie por fuente (tercera pasada, con
             truncado posicional seguro).
          5. Normalización de espacios horizontales (sin tocar \'\\n\').
          6. Regla de oro: nunca más de \'\\n\\n\' seguidos.

        Devuelve (titulo_limpio, contenido_limpio).
        """
        contenido = contenido or ""
        titulo = titulo or ""

        titulo_limpio = clean_titulo(titulo, contenido)

        texto = clean_content(contenido, titulo_limpio)
        texto = clean_content_extra(texto, titulo_limpio)
        texto = clean_content_footer(texto)

        texto = normalizar_espacios_horizontales(texto)
        texto = normalizar_saltos_de_linea(texto)
        texto = texto.strip()

        return titulo_limpio, texto

    def procesar_registro(self, doc: dict) -> dict:
        """Aplica la limpieza a un único artículo (dict), conservando
        intactos el resto de campos/metadatos."""
        titulo_limpio, contenido_limpio = self.limpiar_articulo(
            doc.get("titulo", ""), doc.get("contenido", "")
        )
        nuevo = dict(doc)
        nuevo["titulo"] = titulo_limpio
        nuevo["contenido"] = contenido_limpio
        return nuevo

    def procesar_dataset(self, registros):
        """Procesa una lista completa de artículos.

        Devuelve (registros_procesados, incidencias). Un fallo puntual en
        un registro individual NUNCA detiene el procesamiento del resto: el
        registro problemático se conserva tal cual y la incidencia se
        registra para su revisión posterior."""
        procesados, incidencias = [], []

        for idx, doc in enumerate(registros):
            if not isinstance(doc, dict):
                incidencias.append({
                    "indice": idx, "tipo": "registro_no_es_objeto",
                    "detalle": f"tipo={type(doc).__name__}",
                })
                procesados.append(doc)
                continue
            try:
                procesados.append(self.procesar_registro(doc))
            except Exception as e:
                incidencias.append({
                    "indice": idx,
                    "titulo": str(doc.get("titulo", ""))[:80],
                    "tipo": "fallo_limpieza", "detalle": str(e),
                })
                procesados.append(doc)  # se conserva sin limpiar, mejor que perderlo

        return procesados, incidencias


## 3. Verificación rápida sobre un ejemplo

In [ ]:
limpiador = LimpiadorNoticias()

ejemplo_titulo = "Wall Street revive con la rebaja de S&P un \'lunes negro\'"

# Mojibake real: se genera codificando un texto correcto a UTF-8 y
# decodificándolo como si fuera cp1252 -- el bug real de scraping.
_fragmento_real = "Los mercados —en su conjunto— reaccionaron con fuerte volatilidad tras la rebaja de calificación."
_fragmento_mojibake = _fragmento_real.encode("utf-8").decode("cp1252")

ejemplo_contenido = (
    "Compartir en Twitter\n\n"
    "Wall Street revive con la rebaja de S&P un \u2018lunes negro\u2019\n\n"
    f"{_fragmento_mojibake}\n\n\n\n"
    "## Impacto en Europa\n\n"
    "- Slideshow: Biggest Chapter 11 Cases\n- Eight Tips for Investing\n- Related Story A\n- Related Story B\n\n"
    "Los inversores europeos   reaccionaron    con cautela.\n\n"
    "Copyright © 2011 Reuters"
)

tit, cont = limpiador.limpiar_articulo(ejemplo_titulo, ejemplo_contenido)
print("TÍTULO LIMPIO:")
print(repr(tit))
print()
print("CONTENIDO LIMPIO:")
print(repr(cont))
print()

assert "\n\n\n" not in cont, "quedan 3+ saltos de línea seguidos"
assert "Compartir en Twitter" not in cont, "quedó ruido de compartir en redes"
assert "Copyright" not in cont, "quedó el pie de copyright"
assert "â€" not in cont, "quedó mojibake sin reparar"
assert re.search(r"  ", cont) is None, "quedaron espacios dobles"
print("Verificaciones OK.")


## 4. Procesamiento del dataset completo

Carga `INPUT_JSON` (acepta tanto una lista de artículos como un único artículo),
procesa todos los registros con `LimpiadorNoticias`, guarda el resultado en
`OUTPUT_JSON` y, si estamos en Colab, ofrece la descarga directa del archivo.


In [ ]:
import json

def _cargar_json(path):
    """Carga el JSON de entrada. Acepta tanto una lista de artículos como
    un único artículo (dict); en ese caso lo envuelve en una lista y guarda
    un flag para reconstruir la misma forma al escribir el resultado."""
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, list):
        return data, True
    if isinstance(data, dict):
        return [data], False
    raise ValueError(
        f"Formato de JSON no soportado: se esperaba una lista o un diccionario, "
        f"se encontró {type(data).__name__}."
    )


registros, era_lista = _cargar_json(INPUT_JSON)
print(f"Artículos cargados: {len(registros)}")

procesados, incidencias = limpiador.procesar_dataset(registros)

print(f"Artículos procesados correctamente: {len(procesados) - len(incidencias)}")
print(f"Incidencias registradas:            {len(incidencias)}")

if incidencias:
    print("\nDetalle de incidencias (primeras 20):")
    for e in incidencias[:20]:
        print(" -", e)


In [ ]:
# --- Guardado del resultado -----------------------------------------------------
salida_final = procesados if era_lista else procesados[0]

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(salida_final, f, ensure_ascii=False, indent=2)

print(f"Guardado: {OUTPUT_JSON}  ({len(procesados)} artículos)")

# Informe de incidencias, si las hubo
if incidencias:
    errores_path = OUTPUT_JSON.rsplit(".", 1)[0] + "_errores.json"
    with open(errores_path, "w", encoding="utf-8") as f:
        json.dump(incidencias, f, ensure_ascii=False, indent=2)
    print(f"Detalle de incidencias guardado en: {errores_path}")


## 5. Descarga del archivo limpio

Descarga a tu ordenador el JSON con las noticias ya limpias (y, si hubo alguna
incidencia, también el informe `*_errores.json` correspondiente).


In [ ]:
if IN_COLAB:
    print(f"Descargando {OUTPUT_JSON} ...")
    files.download(OUTPUT_JSON)

    if incidencias:
        errores_path = OUTPUT_JSON.rsplit(".", 1)[0] + "_errores.json"
        print(f"Descargando {errores_path} ...")
        files.download(errores_path)
else:
    print("Fuera de Colab: el archivo ya está guardado en el directorio actual; "
          f"descárgalo directamente desde ahí ({OUTPUT_JSON}).")


## 6. Resumen

| Paso | Función / método | Qué hace |
|---|---|---|
| Mojibake (rápido, best-effort) | `reparar_mojibake_global` | Recodifica todo el texto latin1→utf-8; solo surte efecto si el documento entero está mal codificado |
| Mojibake (robusto) | `reparar_mojibake_fragmentos` | Repara solo los fragmentos con pinta de mojibake, deja el resto intacto |
| Ruido genérico | `clean_content` | Nav bars, muros de suscripción, cookies, bullets de menú, cabeceras de UI, línea de título duplicada |
| Artefactos por portal | `clean_content_extra` | Errores de carga, breadcrumbs de Expansión/El Economista/Cinco Días, live-blogs, notas de prensa pagadas |
| Boilerplate de pie | `clean_content_footer` | Truncado seguro (posición ≥60% del documento) en marcadores de copyright/footer por fuente |
| Espacios horizontales | `normalizar_espacios_horizontales` | Colapsa `" \t"` múltiples a uno solo, sin tocar `\n` |
| **Regla de oro** | `normalizar_saltos_de_linea` | `\n{3,}` → `\n\n` siempre, protegiendo los párrafos para el `SentenceSplitter` |
| Orquestación | `LimpiadorNoticias` | Aplica todo lo anterior por artículo (`procesar_registro`) y por dataset completo (`procesar_dataset`), con manejo de errores por registro |

**Diferencia frente a la versión `.py`:** exactamente la misma lógica de limpieza y la
misma clase `LimpiadorNoticias` — solo cambia la forma de indicar entrada/salida
(variables `INPUT_JSON`/`OUTPUT_JSON` + Drive/`files.upload()`, en vez de `argparse` y
argumentos de terminal), para poder ejecutar el notebook de principio a fin en Colab
sin salir del navegador.
